In [3]:
pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers pypdf fastapi uvicorn flashrank

Note: you may need to restart the kernel to use updated packages.


Goal: Understand how to break down documents so they fit efficiently into Llama 3.1's context window without losing semantic meaning.

**Why?** <br>
* There is a ceiling to the context window
* When you send text to an LLM, the underlying math it has to do to understand that text gets exponentially harder as the text gets longer. This requires VRAM (Video RAM on your graphics card) or regular system RAM.
* The "Lost in the Middle" Phenomenon, where the LLM only looks at the start and end - ignoring the middle

**How does it help?** <br>
1. We Create a Library of Snippets by chunking
2. We scan the library and finds the 3 or 4 index cards that are most mathematically similar to your question (embeddings)
3. Once the system finds those top 3 matching chunks, it grabs only those chunks and pastes them into a prompt that looks like this:
    * "You are a helpful assistant. Please use the following context to answer the user's question:
    * Context: > [Chunk A about refunds]
    * [Chunk B about return shipping]
    * [Chunk C about 30-day guarantees]
    * User Question: What is the company's refund policy?"


In [ ]:
# 1. Create a dummy text file to practice on
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small, 
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.
""" * 10 # Multiplying by 10 to make the text long enough to chunk!

with open("local_rag_guide.txt", "w") as file:
    file.write(sample_text)

# 2. Load the document using LangChain
    # Building a RAG system from scratch requires a lot of repetitive, messy code: opening files, writing loops to break up text, 
    # formatting prompts, sending data to the LLM, and parsing the output. 
    # LangChain provides standardized, pre-built functions for almost everything you need to do with an LLM. 
    # Instead of writing custom code to handle a PDF, a Word Doc, or a web page, LangChain gives you uniform tools to handle them all the exact same way.
from langchain_community.document_loaders import TextLoader

# If you just use standard Python to open a text file, you just get a giant string of text. 
# But in a RAG system, we need to keep track of where that text came from (so we can cite our sources later!).
    # The TextLoader reads your local .txt file and packages it into a standard LangChain Document object organising your data into two parts:
        # page_content: the actual text content of the document
        # metadata: a dictionary where you can store any extra information about the document (like the file path, the title, or the date it was created)
loader = TextLoader("local_rag_guide.txt")
# The .load() method is the trigger.
docs = loader.load()

# 3. Verify it loaded correctly
print(f"Successfully loaded {len(docs)} document(s).")
print(f"Total character count: {len(docs[0].page_content)}")

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 1 document(s).
Total character count: 6160


Now that we have our giant block of text loaded into memory, we need to chop it up. But we can't just slice it blindly by character count—if we do, we might cut a sentence or a word right in half, which will confuse our LLM later.

We want to keep paragraphs and sentences together as much as possible. This is where LangChain's RecursiveCharacterTextSplitter comes in.
* It tries to split the text using a list of separators: like double newlines \n\n for paragraphs, then single newlines \n for sentences, then spaces until the chunks are small enough.

We also use a concept called overlap. By letting the end of one chunk overlap slightly with the beginning of the next, we ensure that we don't accidentally sever the context between two related sentences.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configure the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Maximum number of characters per chunk
    chunk_overlap=50,     # Number of characters to overlap between chunks
    length_function=len,  # How we measure length (standard Python len)
)

# 2. Execute the split on our loaded documents
chunks = text_splitter.split_documents(docs)

# 3. See the results
print(f"Original document was split into {len(chunks)} chunks.")
print("-" * 50)
print("Here is what Chunk 1 looks like:")
print(chunks[0].page_content)
print("-" * 50)
print("Here is what Chunk 2 looks like:")
print(chunks[1].page_content)

Original document was split into 20 chunks.
--------------------------------------------------
Here is what Chunk 1 looks like:
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small,
--------------------------------------------------
Here is what Chunk 2 looks like:
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.


In [6]:
# Bonus: Let's see how different chunking strategies affect the results
# 1. Set up three different chunking strategies
small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
medium_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 2. Apply them to our original loaded documents
small_chunks = small_splitter.split_documents(docs)
medium_chunks = medium_splitter.split_documents(docs)
large_chunks = large_splitter.split_documents(docs)

# 3. Print the results to compare the first chunk of each
print(f"Total Small Chunks: {len(small_chunks)}")
print("--- SMALL CHUNK #1 ---")
print(small_chunks[0].page_content)
print("\n" + "="*50 + "\n")

print(f"Total Medium Chunks: {len(medium_chunks)}")
print("--- MEDIUM CHUNK #1 ---")
print(medium_chunks[0].page_content)
print("\n" + "="*50 + "\n")

print(f"Total Large Chunks: {len(large_chunks)}")
print("--- LARGE CHUNK #1 ---")
print(large_chunks[0].page_content)

Total Small Chunks: 80
--- SMALL CHUNK #1 ---
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between


Total Medium Chunks: 20
--- MEDIUM CHUNK #1 ---
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small,


Total Large Chunks: 10
--- LARGE CHUNK #1 ---
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG